In [1]:
#imports
from delta.tables import DeltaTable, IdentityGenerator
from pyspark.sql.types import LongType, StringType, TimestampType, BooleanType, BinaryType
from datetime import datetime
import ConnectionConfig as cc

debugging_mode=True


In [2]:
#config
cc.setupEnvironment()
spark = cc.startLocalCluster("DIM_USER",4)
spark.getActiveSession()

run_timestamp = datetime.now()

Environment variables are set...


In [3]:
#make connection
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")

In [18]:
#EXTRACT

# user tabel
df_users = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "user_table") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_users.createOrReplaceTempView("users")

# treusure log tabel
df_treasure_log = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure_log") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_treasure_log.createOrReplaceTempView("treasure_log")

#treusure tabel
df_treasure = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_treasure.createOrReplaceTempView("treasure")

#city tabel
df_treasure = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "city") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_treasure.createOrReplaceTempView("city")

#country tabel
df_treasure = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "country") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_treasure.createOrReplaceTempView("country")

INITIAL

In [19]:
#TRANSFORM

#base user
base_user = spark.sql("""
SELECT
        u.id AS userId,
        u.first_name AS first_Name,
        u.last_name AS last_Name,
        u.mail AS email,
        u.city_city_id as cityId,
        CONCAT(u.street, ' ', u.number) AS address,
        COALESCE(tl.treasure_count, 0) AS treasure_count
    FROM users u
    LEFT JOIN (
        SELECT hunter_id, COUNT(*) AS treasure_count
        FROM treasure_log
        GROUP BY hunter_id
    ) tl ON u.id = tl.hunter_id
""")

base_user.createOrReplaceTempView("baseUser")

In [20]:
#TRANSFORM

#expierenceLevel berekenen
expierencelevel_user = spark.sql("""
SELECT
        userId,
        CASE
            WHEN treasure_count = 0 THEN 'Starter'
            WHEN treasure_count < 4 THEN 'Amateur'
            WHEN treasure_count BETWEEN 4 AND 10 THEN 'Professional'
            ELSE 'Pirate'
        END AS experienceLevel
    FROM baseUser
""")
expierencelevel_user.createOrReplaceTempView("expierenceLevelUser")

In [21]:
#TRANSFORM

#dedicator berekenen
dedicator_user = spark.sql("""
   SELECT
        u.userId,
        CASE WHEN COUNT(t.id) > 0 THEN TRUE ELSE FALSE END AS dedicator
    FROM baseUser u
    LEFT JOIN treasure t ON u.userId = t.owner_id
    GROUP BY u.userId
""")
dedicator_user.createOrReplaceTempView("dedicatorUser")

In [22]:
#TRANSFORM

#Country berekenen
country_user = spark.sql("""
    SELECT
        u.userId,
        co.code as country
    FROM baseUser u
    JOIN city ct on ct.city_id = u.cityId
    join country co  on co.code = ct.country_code
""")

country_user.createOrReplaceTempView("countryUser")

In [ ]:
#TRANSFORM

#zet alles samen
complete_user = spark.sql(f"""
SELECT
    b.userId,
    b.first_Name,
    b.last_Name,
    b.email,
    b.address,
    e.experienceLevel,
    d.dedicator,
    c.country,
    to_timestamp('{run_timestamp}') as scd_start,
    to_timestamp(null) AS scd_end,
    TRUE AS current,
    md5(CONCAT(e.experienceLevel,d.dedicator)) AS md5
FROM baseUser b
LEFT JOIN expierenceLevelUser e ON b.userId = e.userId
LEFT JOIN dedicatorUser d ON b.userId = d.userId
LEFT JOIN countryUser c ON b.userId = c.userId
""")

In [ ]:
#LOAD

#maken deltatabel
spark.sql("DROP TABLE IF EXISTS default.dimUser")

DeltaTable.create(spark) \
    .tableName("dimUser") \
    .addColumn("userSurKey", LongType(), nullable=False, generatedAlwaysAs=IdentityGenerator(0, 1)) \
    .addColumn("userId", BinaryType(), nullable=False) \
    .addColumn("first_name", StringType()) \
    .addColumn("last_name", StringType()) \
    .addColumn("email", StringType()) \
    .addColumn("address", StringType()) \
    .addColumn("experiencelevel", StringType()) \
    .addColumn("dedicator", BooleanType()) \
    .addColumn("country", StringType()) \
    .addColumn("scd_start", TimestampType()) \
    .addColumn("scd_end", TimestampType()) \
    .addColumn("md5", StringType()) \
    .addColumn("current", BooleanType()) \
    .property("delta.feature.identityColumns", "supported") \
    .execute()

In [25]:
#LOAD

#write to tabel
complete_user.write.format("delta").mode("overwrite").saveAsTable("dimUser")

In [26]:
#end de spark sessie
spark.stop()